# Verify Generated Annotations

Quick verification pass on the synthetic-conditioned generated images.

Since the annotations come directly from the synthetic generator (not
pseudo-labeling), they should be nearly perfect. This notebook launches the
annotator so you can spot-check and correct any misalignment from the
Gemini style transfer.

**Sections:**
1. Load annotations — inspect `images.json` from the generation step
2. Launch annotator — open the annotation UI for verification
3. Review corrections — inspect annotation stats after review

In [3]:
%load_ext autoreload
%autoreload 2

import json
from pathlib import Path

# ── Configuration ─────────────────────────────────────────────────────────
ANNOTATE_DIR = Path("../data/annotate_generated")
CORRECTED_FILE = ANNOTATE_DIR / "corrected.json"
SERVER_PORT = 7861

print(f"Workspace: {ANNOTATE_DIR.resolve()}")

# Load images.json (written by generate-conditioned CLI)
images_json_path = ANNOTATE_DIR / "images.json"
if images_json_path.exists():
    with open(images_json_path) as f:
        data = json.load(f)
    images_meta = data.get("images", [])
    annotations = data.get("annotations", {})
    n_boxes = sum(len(a["boxes"]) for a in annotations.values())
    print(f"Images: {len(images_meta)}")
    print(f"Annotations: {len(annotations)} images, {n_boxes} boxes total")

    # Board size distribution
    from collections import Counter
    sizes = Counter(img.get("board_size", "?") for img in images_meta)
    print(f"Board sizes: {dict(sizes)}")
else:
    print("No images.json found — run `moku generate-conditioned` first.")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Workspace: /Users/hadim/Code/libs/moku/data/annotate_generated
Images: 1000
Annotations: 1000 images, 146599 boxes total
Board sizes: {19: 703, 9: 155, 13: 142}


## 1. Launch Annotator

Opens the annotation UI to verify and correct the inherited synthetic annotations.

Since annotations come from the synthetic generator, most should be accurate.
Focus on checking:
- Stone positions align with the photorealistic image
- Board corners are correctly placed
- No missing or extra detections

**Tools:** `1` Corner · `2` Black stone · `3` White stone · `V` Move/Pan
**Navigate:** `←`/`→` Prev/Next · `Ctrl+S` Save

> ⚠ Run the cell below and open the printed URL. Press the interrupt button (■) to stop.

In [4]:
import subprocess
import webbrowser

PROJECT_ROOT = Path.cwd().parent
SERVER_SCRIPT = PROJECT_ROOT / "tools" / "annotator" / "server.py"

server_proc = subprocess.Popen(
    ["python", str(SERVER_SCRIPT),
     "--data-dir", str(ANNOTATE_DIR.resolve()),
     "--output", str(CORRECTED_FILE.resolve()),
     "--port", str(SERVER_PORT)],
)

url = f"http://localhost:{SERVER_PORT}"
print(f"Annotator running at {url}")
print("Press the interrupt button (■) to stop the server.")
webbrowser.open(url)

try:
    server_proc.wait()
except KeyboardInterrupt:
    server_proc.terminate()
    print("\nServer stopped.")

Annotator running at http://localhost:7861
Press the interrupt button (■) to stop the server.
Moku Annotator  →  http://localhost:7861
  data-dir  : /Users/hadim/Code/libs/moku/data/annotate_generated
  output    : /Users/hadim/Code/libs/moku/data/annotate_generated/corrected.json
Press Ctrl+C to stop.

Stopped.

Server stopped.


## 2. Review Corrections

Inspect stats on the corrected annotations.

In [6]:
import pandas as pd
from IPython.display import display

CATEGORY_NAMES = {0: "black_stone", 1: "white_stone", 2: "board_corner"}

if not CORRECTED_FILE.exists():
    print(f"No corrections file found at {CORRECTED_FILE}. Run the annotator first.")
else:
    with open(CORRECTED_FILE) as f:
        corrections = json.load(f)

    print(f"Corrections for {len(corrections)} / {len(images_meta)} images")

    # Category breakdown
    cat_counts = {name: 0 for name in CATEGORY_NAMES.values()}
    for fname, corr in corrections.items():
        for box in corr.get("boxes", []):
            cat_name = CATEGORY_NAMES.get(box["category"], "unknown")
            cat_counts[cat_name] = cat_counts.get(cat_name, 0) + 1

    print("\nCategory counts (corrected images):")
    display(pd.Series(cat_counts, name="count").to_frame())

    # Per-image stats
    rows = []
    for fname, corr in corrections.items():
        boxes = corr.get("boxes", [])
        n_corners = sum(1 for b in boxes if b["category"] == 2)
        n_black = sum(1 for b in boxes if b["category"] == 0)
        n_white = sum(1 for b in boxes if b["category"] == 1)
        rows.append({"filename": fname, "corners": n_corners, "black": n_black, "white": n_white})

    df = pd.DataFrame(rows)
    print(f"\nImages with != 4 corners: {(df['corners'] != 4).sum()}")
    display(df.describe())

No corrections file found at ../data/annotate_generated/corrected.json. Run the annotator first.
